# Battery remaining-useful-life playground

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yotambraun/APDTFlow/blob/main/examples/battery_rul_playground.ipynb)

Pick a forecast origin anywhere along a **real NASA battery's** life and see the capacity forecast, the predicted end-of-life (EOL) window, and how the RUL estimate converges as the origin advances.

Every number comes from a model trained on the *other* two cells (**leave-one-battery-out**) — nothing is simulated. Data: NASA Prognostics Data Repository battery set (Saha & Goebel, 2007), cells B0005/B0006/B0007; EOL is the standard 1.4 Ah capacity threshold.

Script twin: [`examples/battery_rul_playground.py`](https://github.com/yotambraun/APDTFlow/blob/main/examples/battery_rul_playground.py). Audited fleet results and the evaluation protocol: [docs/METHODOLOGY.md](https://github.com/yotambraun/APDTFlow/blob/main/docs/METHODOLOGY.md), [docs/experiment_results.md](https://github.com/yotambraun/APDTFlow/blob/main/docs/experiment_results.md).

In [ ]:
%pip install -q apdtflow ipywidgets

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from apdtflow import APDTFlowForecaster, set_seed

set_seed(0)

CELLS = ['B0005', 'B0006', 'B0007']
EOL_THRESHOLD = 1.4  # Ah — standard battery end-of-life definition
HISTORY, HORIZON = 30, 30
EPOCHS = 15


def load_capacity(cell):
    """Per-cycle capacity (Ah) from GitHub, falling back to a local checkout."""
    url = ('https://raw.githubusercontent.com/yotambraun/APDTFlow/main/'
           f'dataset_examples/nasa_{cell}.csv')
    local = Path('dataset_examples') / f'nasa_{cell}.csv'
    try:
        df = pd.read_csv(url)
    except Exception:  # offline / local checkout
        df = pd.read_csv(local)
    return df['capacity'].to_numpy(dtype=float)


capacity_by_cell = {c: load_capacity(c) for c in CELLS}
for c, cap in capacity_by_cell.items():
    print(f'{c}: {len(cap)} cycles, capacity {cap.max():.2f} -> {cap.min():.2f} Ah')

## Train leave-one-battery-out

We hold out one cell entirely and train on the other two. `decoder_type='continuous'` enables `predict_at`/`predict_when`; `use_conformal=True` calibrates the EOL time window (here at `alpha=0.1`, a 90% window).

In [ ]:
HELD_OUT = 'B0005'  # the cell we explore; change to B0006 or B0007 and re-run

train_series = np.concatenate(
    [capacity_by_cell[c] for c in CELLS if c != HELD_OUT]
)
model = APDTFlowForecaster(
    forecast_horizon=HORIZON,
    history_length=HISTORY,
    num_epochs=EPOCHS,
    decoder_type='continuous',
    use_conformal=True,
    verbose=False,
)
model.fit(pd.DataFrame({'capacity': train_series}), target_col='capacity')

capacity = capacity_by_cell[HELD_OUT]
print(f'Trained on the other two cells; exploring held-out {HELD_OUT} ({len(capacity)} cycles).')

## Forecast from any origin

`show_origin(origin)` forecasts forward from measurement `origin` of the held-out cell: the continuous capacity trajectory (`predict_at`), the calibrated 90% EOL window (`predict_when`), and what actually happened.

The model was fitted on the training cells; to forecast the held-out cell from an arbitrary point we hand it that cell's most recent window (normalized with the *training* statistics — the same no-leakage pattern as `historical_forecasts`).

In [ ]:
def forecast_from(origin):
    """Point the fitted model at the held-out cell's window ending at `origin`."""
    window = capacity[origin - HISTORY:origin]
    model.last_sequence_ = (window - model.scaler_mean_) / model.scaler_std_


def show_origin(origin):
    origin = int(np.clip(origin, HISTORY, len(capacity) - 1))
    forecast_from(origin)

    taus = np.linspace(0.5, HORIZON, 120)
    traj, _, _ = model.predict_at(taus)
    result = model.predict_when(EOL_THRESHOLD, direction='below', alpha=0.1)

    future = capacity[origin:]
    hit = np.nonzero(future < EOL_THRESHOLD)[0]
    actual_rul = int(hit[0]) if len(hit) else None

    print(f'{HELD_OUT}, origin at measurement #{origin} '
          f'(capacity {capacity[origin - 1]:.3f} Ah)')
    if result.censored:
        print(f'  predicted: no EOL crossing within {HORIZON} measurements (censored)')
    else:
        print(f'  predicted EOL in {result.eta:.1f} measurements '
              f'(90% window {result.earliest:.1f} .. {result.latest:.1f}, '
              f'act_by {result.act_by:.1f}  <- schedule by this, never by eta)')
    print('  actual:    ' + (f'EOL in {actual_rul} measurements'
                             if actual_rul is not None else 'no EOL in the data'))

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(np.arange(len(capacity)), capacity, color='gray', lw=1, label='capacity')
    ax.plot(origin + taus, traj, color='C0', lw=2, label='forecast')
    ax.axhline(EOL_THRESHOLD, color='C3', lw=1, label=f'EOL {EOL_THRESHOLD} Ah')
    if not result.censored:
        ax.axvspan(origin + result.earliest, origin + result.latest,
                   color='C2', alpha=0.2, label='90% EOL window')
    if actual_rul is not None:
        ax.plot(origin + actual_rul, EOL_THRESHOLD, 'k*', ms=14, label='actual EOL')
    ax.axvline(origin, color='k', ls=':', lw=1)
    ax.set_title(f'{HELD_OUT}: forecast from origin #{origin}')
    ax.set_xlabel('measurement #')
    ax.set_ylabel('capacity (Ah)')
    ax.legend(fontsize=8)
    plt.show()

In [ ]:
# Interactive origin slider when ipywidgets is available; plain variable otherwise.
try:
    from ipywidgets import IntSlider, interact

    interact(show_origin, origin=IntSlider(
        value=max(HISTORY, len(capacity) // 2),
        min=HISTORY, max=len(capacity) - 1, step=5,
        description='origin', continuous_update=False,
    ))
except ImportError:
    ORIGIN = max(HISTORY, len(capacity) // 2)  # <- edit and re-run
    show_origin(ORIGIN)

## RUL convergence as the origin advances

Slide the origin along the whole life of the cell: early on, EOL is beyond the horizon (censored — a valid answer, not a failure); as degradation approaches the threshold, the predicted RUL should converge toward the actual one.

In [ ]:
origins = list(range(HISTORY, len(capacity) - 1, 5))
pred_rul, actual_rul_curve = [], []
for o in origins:
    forecast_from(o)
    r = model.predict_when(EOL_THRESHOLD, direction='below', alpha=0.1)
    pred_rul.append(np.nan if r.censored else float(r.eta))
    fut = capacity[o:]
    hit = np.nonzero(fut < EOL_THRESHOLD)[0]
    actual_rul_curve.append(float(hit[0]) if len(hit) else np.nan)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(origins, actual_rul_curve, 'k--', lw=1, label='actual RUL')
ax.plot(origins, pred_rul, 'o-', color='C0', ms=3, lw=1, label='predicted RUL (eta)')
ax.set_title(f'{HELD_OUT}: RUL convergence as the origin advances '
             f'(blank = censored / no EOL within horizon)')
ax.set_xlabel('forecast origin (measurement #)')
ax.set_ylabel('measurements until EOL')
ax.legend(fontsize=8)
plt.show()

## Notes

- **Schedule by `act_by`** (the window's early edge), never by the point estimate `eta` — the point estimate runs systematically late on smoothed degradation indicators; the asymmetric time-space calibration absorbs that bias.
- `alpha` is the sensitivity knob: smaller alpha widens the window and moves `act_by` earlier (fewer missed events, more false alarms).
- The audited leave-one-battery-out results behind this demo, and the baseline protocol every published number must pass, are in [docs/METHODOLOGY.md](https://github.com/yotambraun/APDTFlow/blob/main/docs/METHODOLOGY.md) and [docs/experiment_results.md](https://github.com/yotambraun/APDTFlow/blob/main/docs/experiment_results.md).
- Whole fleets: `model.predict_when_fleet({asset_id: series, ...}, threshold, direction='below')` returns a maintenance schedule sorted by `act_by`.